# TimesFM 2.5 — Wind
Zero-shot rolling forecasts. The covariate variant uses inference-time regression support.

This notebook reports the overall MAE, RMSE, and aQL result.


In [ ]:
# !pip install -r requirements/timesfm.txt


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "requirements").exists():
    REPO_ROOT = REPO_ROOT.parent
ROOT = REPO_ROOT / "wind"
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from preprocessing import load_wind, describe_wind, NWP_FEATURES, TEST_START, SEED, CONTEXT_LENGTH, PREDICTION_LENGTH, SITES
from src.metrics import print_metrics
from power_conversion import convert_wind_result_to_power

BASE_QUANTILES=[0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9]
TAIL_QUANTILES=[0.01,0.025,0.05,0.95,0.975,0.99]
OUTPUT_QUANTILES=sorted(set(BASE_QUANTILES+TAIL_QUANTILES))
QUANTILES=BASE_QUANTILES
MODEL_QUANTILES=BASE_QUANTILES
OUTPUT_DIR=ROOT/"outputs"/"predictions"; OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
data = load_wind(ROOT / "data" / "wind")
print(describe_wind(data))

from tqdm.auto import tqdm
import timesfm


In [ ]:
VARIANT = "with_covariates"
model = timesfm.TimesFM_2p5_200M_torch.from_pretrained("google/timesfm-2.5-200m-pytorch")
model.compile(timesfm.ForecastConfig(max_context=CONTEXT_LENGTH, max_horizon=744, normalize_inputs=True, use_continuous_quantile_head=True, force_flip_invariance=True, infer_is_positive=True, fix_quantile_crossing=True, return_backcast=True))


In [ ]:
rolling = {s: data.context_by_site[s].copy() for s in SITES}
rows=[]
for i in tqdm(range(len(data.daily_by_site[SITES[0]]))):
    day={s:data.daily_by_site[s][i][1].copy() for s in SITES}
    for site in SITES:
        context=rolling[site].sort_values("timestamp")
        future=day[site].sort_values("timestamp")
        hist=context.target.to_numpy()[-CONTEXT_LENGTH:]
        if VARIANT == "with_covariates":
            dynamic={}
            for k,feature in enumerate(NWP_FEATURES,1):
                dynamic[f"Exogenous {k}"]=[np.concatenate([context[feature].to_numpy()[-CONTEXT_LENGTH:], future[feature].to_numpy()])]
            point,qhat=model.forecast_with_covariates(inputs=[hist], dynamic_numerical_covariates=dynamic, dynamic_categorical_covariates={}, static_numerical_covariates={}, static_categorical_covariates={}, xreg_mode="timesfm + xreg", ridge=0.0, normalize_xreg_target_per_input=True)
        else:
            point,qhat=model.forecast(horizon=PREDICTION_LENGTH, inputs=[hist])
        point = np.asarray(point)
        qhat = np.asarray(qhat)

        point = point[0, -PREDICTION_LENGTH:]
        qhat = qhat[0, -PREDICTION_LENGTH:]
        for h,ts in enumerate(future.timestamp.iloc[:PREDICTION_LENGTH]):
            row={"item_id":int(future.item_id.iloc[0]),"timestamp":ts,"prediction":float(point[h])}
            for qi,q in enumerate(MODEL_QUANTILES): row[str(q)]=float(qhat[h,qi+1])
            rows.append(row)
    for site in SITES: rolling[site]=pd.concat([rolling[site],day[site]],ignore_index=True)
result=pd.DataFrame(rows).merge(data.test[["item_id","timestamp","target"]],on=["item_id","timestamp"],how="inner")

# Keep a common 15-quantile CSV schema. Unsupported model quantiles remain NaN.
for _q in OUTPUT_QUANTILES:
    if str(_q) not in result.columns:
        result[str(_q)] = np.nan
result.to_csv(OUTPUT_DIR / f"timesfm_{VARIANT}.csv", index=False)
print_metrics(result, QUANTILES, label=f"timesfm_{VARIANT}")

# Wind speed -> wind power evaluation
power_result=convert_wind_result_to_power(result)

# Keep a common 15-quantile CSV schema. Unsupported model quantiles remain NaN.
for _q in OUTPUT_QUANTILES:
    if str(_q) not in power_result.columns:
        power_result[str(_q)] = np.nan
power_result.to_csv(OUTPUT_DIR/f"timesfm_{VARIANT}_power.csv",index=False)
print_metrics(power_result,QUANTILES,label=f"timesfm_{VARIANT} [wind power]")
